In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from shapely.geometry import Point, Polygon
from scipy.spatial import ConvexHull


In [6]:
excel_file_path1 = 'C:/Users/ASUS/OneDrive/Documents/中信兄弟/Season_data/2024_non_trackman.xlsx'
excel_file_path2 = 'C:/Users/ASUS/OneDrive/Documents/中信兄弟/Season_data/2023_non_trackman.xlsx'
excel_file_path3 = 'C:/Users/ASUS/OneDrive/Documents/中信兄弟/Season_data/2025_non_trackman.xlsx'
sheet_name = 'Sheet1'

In [7]:
data2024 = pd.read_excel(excel_file_path1, sheet_name=sheet_name)
data2023 = pd.read_excel(excel_file_path2, sheet_name=sheet_name)
data2025 = pd.read_excel(excel_file_path3, sheet_name=sheet_name)

In [8]:
col = ['umpireInChief','date','stadium','bHand','pHand','strikes','balls','side','inning','coordX','coordY','pType','call','result']

In [9]:
data2024= data2024[col]
data2023= data2023[col]
data2025= data2025[col]

In [10]:
df = pd.concat([data2024, data2025,data2023], ignore_index=True)

In [11]:
df.loc[:, 'Year'] = df['date'].str[:4]

In [12]:
df = df[~df['result'].isin(['GO','2B', '1B', 'FO', 'SH', 'E', 'GIDP', 'FC','HBP', 'HR', '3B', 'IBB', 'SF', 'IGNORE', 'DP', 'D3S', 'ID', 'IH'])]
df = df[~df['call'].isin(['F', 'SW', 'CS', 'FT_MISS', 'FOUL_BUNT','FT','TRY_BUNT', 'H', 'BUNT'])]
df= df.dropna(subset=['call'])

In [13]:
name_mapping = {
    'FF':'直球', 
    'CH':'變速球', 
    'FC':'卡特球', 
    'CU':'曲球', 
    'FO':'指叉球',
    'SL':'滑球', 
    'SI':'伸卡球',
    'KN':'彈指曲球', 
    'EP':'小便球'
}

# 新增一列"球員"，根据"Pitcher"列的值匹配中文名
df['球種'] = df['pType'].map(name_mapping)

In [14]:
# 判斷投球是否在規則書好球帶內
def in_strike_zone(x, y):
    return -50 <= x <= 50 and -50 <= y <= 50

df["rulebook_strike"] = df.apply(lambda row: in_strike_zone(row["coordX"], row["coordY"]), axis=1)

In [17]:
df

,umpireInChief,date,stadium,bHand,pHand,strikes,balls,side,inning,coordX,coordY,pType,call,result,Year,球種,rulebook_strike
1,羅鈞鴻,2024-10-15T06:25:07.183Z,樂天桃園棒球場 Rakuten Taoyuan Baseball Stadium,L,R,1,0,AWAY,1,55.45,36.38,FC,S,NaN,2024,卡特球,False
3,羅鈞鴻,2024-10-15T06:25:07.183Z,樂天桃園棒球場 Rakuten Taoyuan Baseball Stadium,L,R,2,0,AWAY,1,-70.72,-158.36,FO,B,NaN,2024,指叉球,False
7,羅鈞鴻,2024-10-15T06:25:07.183Z,樂天桃園棒球場 Rakuten Taoyuan Baseball Stadium,R,R,0,0,AWAY,1,25.28,-51.39,FF,S,NaN,2024,直球,False
9,羅鈞鴻,2024-10-15T06:25:07.183Z,樂天桃園棒球場 Rakuten Taoyuan Baseball Stadium,R,R,2,0,AWAY,1,-88.55,-88.42,FC,B,NaN,2024,卡特球,False
10,羅鈞鴻,2024-10-15T06:25:07.183Z,樂天桃園棒球場 Rakuten Taoyuan Baseball Stadium,R,R,2,1,AWAY,1,-57.01,-119.96,SL,B,NaN,2024,滑球,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
299124,吳家維,2023-10-25T10:35:00.000Z,嘉義市立棒球場,R,R,0,0,AWAY,9,-5.99,-128.20,CU,B,NaN,2023,曲球,False
299129,吳家維,2023-10-25T10:35:00.000Z,嘉義市立棒球場,R,R,0,0,AWAY,9,-78.11,44.07,CU,B,NaN,2023,曲球,False
299130,吳家維,2023-10-25T10:35:00.000Z,嘉義市立棒球場,R,R,0,1,AWAY,9,-36.04,-30.05,FF,B,NaN,2023,直球,True
299131,吳家維,2023-10-25T10:35:00.000Z,嘉義市立棒球場,R,R,0,2,AWAY,9,28.06,-2.00,CU,S,NaN,2023,曲球,True


In [30]:
from shapely.geometry import Point, Polygon
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde
from scipy.spatial import ConvexHull

# ----------------------------
# 建立裁判實際好球帶（KDE）
# ----------------------------
def build_umpire_zone(df_ump, bandwidth=12, threshold=0.35, grid_size=300, outlier_std=4.0):
    strikes = df_ump[df_ump["call"] == "S"][["coordX", "coordY"]].dropna().values
    if len(strikes) < 3:
        return None

    mean = np.mean(strikes, axis=0)
    std = np.std(strikes, axis=0)
    std[std == 0] = 1e-6  # 防止除0
    z_score = np.abs((strikes - mean) / std)
    strikes = strikes[(z_score < outlier_std).all(axis=1)]

    if len(strikes) < 3:
        strikes = df_ump[df_ump["call"] == "S"][["coordX", "coordY"]].dropna().values  # fallback

    kde = gaussian_kde(strikes.T, bw_method='scott')

    x_min, x_max = strikes[:, 0].min() - 20, strikes[:, 0].max() + 20
    y_min, y_max = strikes[:, 1].min() - 20, strikes[:, 1].max() + 20
    x, y = np.meshgrid(np.linspace(x_min, x_max, grid_size),
                       np.linspace(y_min, y_max, grid_size))
    xy = np.vstack([x.ravel(), y.ravel()])
    z = kde(xy).reshape(grid_size, grid_size)
    z = z / z.max()
    mask = z >= threshold
    pts = np.column_stack([x[mask], y[mask]])
    
    if len(pts) < 3:
        pts = strikes
    hull = ConvexHull(pts)
    zone = Polygon(pts[hull.vertices])
    return zone if zone.is_valid else None


# ----------------------------
# 評估裁判每顆球的正確性與一致性
# ----------------------------
def evaluate_calls(df_ump, ump_zone, true_zone):
    results = []
    for _, row in df_ump.iterrows():
        pt = Point(row["coordX"], row["coordY"])
        inside_true = true_zone.contains(pt)
        inside_ump = ump_zone.contains(pt) if ump_zone else False

        correct = (inside_true and row["call"] == "S") or (not inside_true and row["call"] == "B")
        consistent = (inside_ump and row["call"] == "S") or (not inside_ump and row["call"] == "B")

        if inside_true and row["call"] == "B":
            miss_type = "TrueStrike_CalledBall"
        elif (not inside_true) and row["call"] == "S":
            miss_type = "TrueBall_CalledStrike"
        else:
            miss_type = "Correct"

        results.append((correct, consistent, miss_type))

    df_ump[["Correct", "Consistent", "MissType"]] = results
    return df_ump


# ----------------------------
# 計算裁判一致性（整體 + 左右打者）
# ----------------------------
def compute_umpire_consistency(df, true_zone, bandwidth=12, threshold=0.35):
    results = []

    for umpire, df_u in df.groupby("umpireInChief"):
        # ---------- 整體 ----------
        zone_all = build_umpire_zone(df_u, bandwidth, threshold)
        df_eval = evaluate_calls(df_u.copy(), zone_all, true_zone)
        overall_acc = df_eval["Correct"].mean()
        overall_con = df_eval["Consistent"].mean()

        # ---------- 左右打 ----------
        zones = {}
        acc_lr = {}
        con_lr = {}

        for hand, df_hand in df_u.groupby("bHand"):
            zone = build_umpire_zone(df_hand, bandwidth, threshold)
            zones[hand] = zone

            if zone is not None:
                df_eval_hand = evaluate_calls(df_hand.copy(), zone, true_zone)
                acc_lr[hand] = df_eval_hand["Correct"].mean()
                con_lr[hand] = df_eval_hand["Consistent"].mean()
            else:
                # fallback: 用整體區域計算
                if zone_all is not None:
                    df_eval_hand = evaluate_calls(df_hand.copy(), zone_all, true_zone)
                    acc_lr[hand] = df_eval_hand["Correct"].mean()
                    con_lr[hand] = df_eval_hand["Consistent"].mean()
                else:
                    acc_lr[hand] = np.nan
                    con_lr[hand] = np.nan

        # ---------- 左右打重疊一致性 ----------
        overlap_percent = np.nan
        if "L" in zones and "R" in zones and zones["L"] and zones["R"]:
            try:
                inter_area = zones["L"].intersection(zones["R"]).area
                union_area = zones["L"].union(zones["R"]).area
                if union_area > 0:
                    overlap_percent = (inter_area / union_area) * 100
            except:
                overlap_percent = np.nan

        results.append({
            "裁判": umpire,
            "整體準確率": round(overall_acc * 100, 2),
            "整體一致性": round(overall_con * 100, 2),
            "對左打_準確率": round(acc_lr.get("L", np.nan) * 100, 2),
            "對左打_一致性": round(con_lr.get("L", np.nan) * 100, 2),
            "對右打_準確率": round(acc_lr.get("R", np.nan) * 100, 2),
            "對右打_一致性": round(con_lr.get("R", np.nan) * 100, 2),
            "左右打重疊比率(%)": round(overlap_percent, 2)
        })

    df_summary = pd.DataFrame(results).sort_values("整體一致性", ascending=False)
    return df_summary

#LR_Overlap(%):同一位裁判在左打者 (L) 與右打者 (R) 的好球帶重疊程度（百分比）越高=左右打者好球帶形狀與位置幾乎相同

In [31]:
# 定義規則好球帶（例如寬 100、高 120 的矩形）
true_zone = Polygon([
    (-50, -50),
    (-50, 50),
    (50, 50),
    (50, -50)
])

# 計算結果
df_result = compute_umpire_consistency(df, true_zone, bandwidth=12, threshold=0.35)
df_result

C:\Users\ASUS\anaconda3\Lib\site-packages\shapely\predicates.py:526: RuntimeWarning: invalid value encountered in contains
  return lib.contains(a, b, **kwargs)
C:\Users\ASUS\anaconda3\Lib\site-packages\shapely\predicates.py:526: RuntimeWarning: invalid value encountered in contains
  return lib.contains(a, b, **kwargs)
C:\Users\ASUS\anaconda3\Lib\site-packages\shapely\predicates.py:526: RuntimeWarning: invalid value encountered in contains
  return lib.contains(a, b, **kwargs)
C:\Users\ASUS\anaconda3\Lib\site-packages\shapely\predicates.py:526: RuntimeWarning: invalid value encountered in contains
  return lib.contains(a, b, **kwargs)
C:\Users\ASUS\anaconda3\Lib\site-packages\shapely\predicates.py:526: RuntimeWarning: invalid value encountered in contains
  return lib.contains(a, b, **kwargs)
C:\Users\ASUS\anaconda3\Lib\site-packages\shapely\predicates.py:526: RuntimeWarning: invalid value encountered in contains
  return lib.contains(a, b, **kwargs)
C:\Users\ASUS\anaconda3\Lib\site-p

,裁判,整體準確率,整體一致性,對左打_準確率,對左打_一致性,對右打_準確率,對右打_一致性,左右打重疊比率(%)
15,陳乃瑞,89.54,93.53,88.84,92.56,90.14,93.36,64.17
0,劉世偉,87.81,92.19,87.95,90.51,87.65,92.76,67.04
4,彭楚雲,88.97,91.53,89.71,91.69,88.11,90.53,74.71
12,蔡豐澤,88.40,91.46,88.96,91.12,87.68,90.79,67.08
1,吳家維,89.64,91.22,90.13,91.06,89.10,91.32,68.63
9,紀華文,89.26,91.19,89.72,91.23,88.71,90.12,68.53
5,林金達,87.49,91.15,87.49,91.16,87.49,90.83,65.17
14,邱景彥,88.95,91.14,88.31,90.29,89.65,91.30,71.53
6,楊崇煇,88.48,91.12,88.34,90.68,88.66,90.81,77.97
3,張展榮,88.15,91.08,88.73,90.30,87.53,90.14,62.88


In [37]:
def compute_umpire_consistency_by_year(df, true_zone, bandwidth=12, threshold=0.35):
    results = []

    # 逐年統計
    for year, df_year in df.groupby("Year"):
        for umpire, df_u in df_year.groupby("umpireInChief"):
            # ---------- 整體 ----------
            zone_all = build_umpire_zone(df_u, bandwidth, threshold)
            df_eval = evaluate_calls(df_u.copy(), zone_all, true_zone)
            overall_acc = df_eval["Correct"].mean()
            overall_con = df_eval["Consistent"].mean()

            # ---------- 左右打 ----------
            zones = {}
            acc_lr = {}
            con_lr = {}

            for hand, df_hand in df_u.groupby("bHand"):
                zone = build_umpire_zone(df_hand, bandwidth, threshold)
                zones[hand] = zone

                if zone is not None:
                    df_eval_hand = evaluate_calls(df_hand.copy(), zone, true_zone)
                    acc_lr[hand] = df_eval_hand["Correct"].mean()
                    con_lr[hand] = df_eval_hand["Consistent"].mean()
                else:
                    # fallback 用整體區域
                    if zone_all is not None:
                        df_eval_hand = evaluate_calls(df_hand.copy(), zone_all, true_zone)
                        acc_lr[hand] = df_eval_hand["Correct"].mean()
                        con_lr[hand] = df_eval_hand["Consistent"].mean()
                    else:
                        acc_lr[hand] = np.nan
                        con_lr[hand] = np.nan

            # ---------- 左右打重疊一致性 ----------
            overlap_percent = np.nan
            if "L" in zones and "R" in zones and zones["L"] and zones["R"]:
                try:
                    inter_area = zones["L"].intersection(zones["R"]).area
                    union_area = zones["L"].union(zones["R"]).area
                    if union_area > 0:
                        overlap_percent = (inter_area / union_area) * 100
                except:
                    overlap_percent = np.nan

            results.append({
                "Year": year,
                "裁判": umpire,
                "整體準確率": round(overall_acc * 100, 2),
                "整體一致性": round(overall_con * 100, 2),
                "對左打_準確率": round(acc_lr.get("L", np.nan) * 100, 2),
                "對左打_一致性": round(con_lr.get("L", np.nan) * 100, 2),
                "對右打_準確率": round(acc_lr.get("R", np.nan) * 100, 2),
                "對右打_一致性": round(con_lr.get("R", np.nan) * 100, 2),
                "左右打重疊比率(%)": round(overlap_percent, 2)
            })

    df_summary = pd.DataFrame(results)
    df_summary = df_summary.sort_values(["Year", "整體一致性"], ascending=[True, False])
    return df_summary

# 定義規則好球帶（例如寬 100、高 120 的矩形）
true_zone = Polygon([
    (-50, -50),
    (-50, 50),
    (50, 50),
    (50, -50)
])

# 計算結果
df_result_by_year = compute_umpire_consistency_by_year(df, true_zone, bandwidth=12, threshold=0.35)
df_result_by_year


C:\Users\ASUS\anaconda3\Lib\site-packages\shapely\predicates.py:526: RuntimeWarning: invalid value encountered in contains
  return lib.contains(a, b, **kwargs)
C:\Users\ASUS\anaconda3\Lib\site-packages\shapely\predicates.py:526: RuntimeWarning: invalid value encountered in contains
  return lib.contains(a, b, **kwargs)
C:\Users\ASUS\anaconda3\Lib\site-packages\shapely\predicates.py:526: RuntimeWarning: invalid value encountered in contains
  return lib.contains(a, b, **kwargs)
C:\Users\ASUS\anaconda3\Lib\site-packages\shapely\predicates.py:526: RuntimeWarning: invalid value encountered in contains
  return lib.contains(a, b, **kwargs)
C:\Users\ASUS\anaconda3\Lib\site-packages\shapely\predicates.py:526: RuntimeWarning: invalid value encountered in contains
  return lib.contains(a, b, **kwargs)
C:\Users\ASUS\anaconda3\Lib\site-packages\shapely\predicates.py:526: RuntimeWarning: invalid value encountered in contains
  return lib.contains(a, b, **kwargs)
C:\Users\ASUS\anaconda3\Lib\site-p

,Year,裁判,整體準確率,整體一致性,對左打_準確率,對左打_一致性,對右打_準確率,對右打_一致性,左右打重疊比率(%)
2,2023,張展榮,87.71,91.56,87.59,88.81,87.85,91.17,55.17
5,2023,楊崇煇,87.77,91.50,88.04,89.69,87.46,91.45,71.79
3,2023,彭楚雲,88.31,91.42,89.04,91.68,87.38,91.28,72.79
11,2023,蔡豐澤,88.92,91.16,89.67,91.50,87.98,90.14,57.32
13,2023,邱景彥,88.21,91.03,88.04,90.61,88.40,90.28,64.47
8,2023,紀華文,88.26,90.59,89.22,91.50,86.95,89.36,60.74
4,2023,林金達,86.06,90.58,85.66,90.47,86.50,89.96,58.05
0,2023,吳家維,89.78,90.09,90.64,90.34,88.93,90.80,60.46
1,2023,尤志欽,86.92,90.07,86.38,89.29,87.56,90.85,65.53
7,2023,王俊宏,86.74,89.85,86.55,89.38,86.94,88.72,56.09


In [38]:
#df_result_by_year.to_excel('裁判歷年準確率,一致性.xlsx', index=True)